# Projekt końcowy – Analiza sygnałów EKG
## Zadanie 3: Przewidywanie zaburzeń rytmu serca (LSTM)
## Zadanie 4: Wykrywanie anomalii rytmu serca (Autoenkoder)
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, roc_curve, auc,
    accuracy_score, precision_score, recall_score, f1_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from tensorflow.keras.utils import plot_model

import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow version:', tf.__version__)
print('NumPy version:', np.__version__)
print('Pandas version:', pd.__version__)

---
# Zadanie 3 – Przewidywanie zaburzeń rytmu serca (LSTM)
---
## Krok 1 – Wczytanie i eksploracja danych

In [ ]:
# Wczytanie danych – brak nagłówka, etykieta oddzielona dwukropkiem w ostatniej kolumnie
# Wczytujemy jako tekst, potem rozdzielamy ostatnią kolumnę
raw = pd.read_csv(
    'ECG5000.csv',
    header=None,
    sep=',',
    converters={140: lambda x: x.split(':')[-1]}  # ostatnia kolumna: 'wartość:etykieta'
)

# Ostatnia kolumna zawiera łańcuch 'wartość:etykieta' – rozdzielamy ją
last_col = raw.iloc[:, 140].astype(str)
# Sprawdzamy czy zawiera dwukropek
print(last_col.head())

In [ ]:
# Właściwe wczytanie: każda linia to 140 wartości + ':etykieta' w ostatnim polu
# Użyjemy własnego parsera

rows = []
with open('ECG5000.csv', 'r') as f:
    for line in f:
        parts = line.strip().split(',')
        # ostatni element ma postać 'float:int'
        last = parts[-1]
        last_val, label = last.rsplit(':', 1)
        row = [float(p) for p in parts[:-1]] + [float(last_val), int(label)]
        rows.append(row)

cols = [f'v{i}' for i in range(140)] + ['type_orig']
df = pd.DataFrame(rows, columns=cols)

# Kodowanie: 1 (Normal) -> 0, pozostałe -> 1
df['type'] = df['type_orig'].apply(lambda x: 0 if x == 1 else 1)
df.drop(columns=['type_orig'], inplace=True)

print('Kształt ramki danych:', df.shape)
print()
print(df.info())

In [ ]:
print('Pierwsze 5 wierszy:')
display(df.head())
print('Ostatnie 5 wierszy:')
display(df.tail())

In [ ]:
# Proporcje klas
class_counts = df['type'].value_counts()
class_labels = {0: 'Normalny (0)', 1: 'Arytmia (1)'}
print('Proporcje klas:')
print(class_counts)
print()
print('Proporcje procentowe:')
print(class_counts / len(df) * 100)

# Wykres kołowy
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    class_counts,
    labels=[class_labels[i] for i in class_counts.index],
    autopct='%1.1f%%',
    startangle=90,
    colors=['#4CAF50', '#F44336'],
    explode=(0.05, 0.05)
)
ax.set_title('Proporcje klas: Normalny vs Arytmia', fontsize=14)
plt.tight_layout()
plt.show()

### Jakie metryki będą miarodajne?

Dane są **niezbalansowane** – obserwacji normalnych jest znacznie więcej niż anomalnych.
W takim przypadku sama dokładność (accuracy) może być myląca (model przewidujący zawsze klasę 0 osiągnąłby wysoką accuracy).

**Miarodajne metryki:**
- **Recall (czułość / TPR)** – zdolność do wykrywania arytmii (istotna medycznie, bo pominięcie arytmii jest groźne)
- **Precision** – precyzja detekcji arytmii
- **F1-score** – harmonijna średnia precision i recall
- **AUC-ROC** – miara jakości separacji klas, odporna na nierównowagę
- **Macierz pomyłek** – pełny obraz TP, FP, TN, FN
- **Accuracy** – pomocnicza, o ile weźmiemy pod uwagę niezbalansowanie

In [ ]:
# Wykresy EKG – zdrowi i z arytmią
normal_samples = df[df['type'] == 0].iloc[:5, :140].values
anom_samples   = df[df['type'] == 1].iloc[:5, :140].values

fig, axes = plt.subplots(5, 1, figsize=(12, 10), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(normal_samples[i], color='#2196F3', linewidth=1)
    ax.set_ylabel(f'Próbka {i+1}')
    ax.grid(True, alpha=0.3)
axes[0].set_title('EKG – osoby zdrowe (klasa 0)', fontsize=13)
axes[-1].set_xlabel('Czas [próbki]')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(5, 1, figsize=(12, 10), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(anom_samples[i], color='#F44336', linewidth=1)
    ax.set_ylabel(f'Próbka {i+1}')
    ax.grid(True, alpha=0.3)
axes[0].set_title('EKG – osoby z arytmią (klasa 1)', fontsize=13)
axes[-1].set_xlabel('Czas [próbki]')
plt.tight_layout()
plt.show()

## Krok 2 – Przygotowanie danych dla LSTM

In [ ]:
# Podział na X i y
feature_cols = [c for c in df.columns if c != 'type']
X = df[feature_cols].values
y = df['type'].values

# Podział treningowy/testowy 70:30
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print('X_train:', X_train.shape, '  X_test:', X_test.shape)
print('y_train:', y_train.shape, '  y_test:', y_test.shape)

# Normalizacja do [0, 1]
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Rozszerzenie wymiaru do 3D: (próbki, czas, cechy=1)
X_train_3d = X_train_scaled[:, :, np.newaxis]
X_test_3d  = X_test_scaled[:, :, np.newaxis]

print()
print('X_train_3d:', X_train_3d.shape)
print('X_test_3d: ', X_test_3d.shape)
print('y_train:   ', y_train.shape)
print('y_test:    ', y_test.shape)

## Krok 3 – Budowa modelu LSTM

In [ ]:
n_timesteps = X_train_3d.shape[1]  # 140
n_features  = X_train_3d.shape[2]  # 1

model_lstm = keras.Sequential([
    keras.Input(shape=(n_timesteps, n_features)),
    layers.LSTM(100, return_sequences=True, name='lstm_1'),
    layers.LSTM(100, return_sequences=False, name='lstm_2'),
    layers.Dropout(0.3, name='dropout'),
    layers.Dense(1, activation='sigmoid', name='output')
], name='LSTM_Arytmia')

model_lstm.summary()

## Krok 4 – Kompilacja, wizualizacja i trening

In [ ]:
model_lstm.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Wizualizacja modelu
try:
    plot_model(model_lstm, show_shapes=True, show_layer_names=True,
               to_file='lstm_model.png', dpi=80)
    from IPython.display import Image
    display(Image('lstm_model.png'))
except Exception:
    print('Wizualizacja modelu niedostępna (brak graphviz). Podsumowanie powyżej.')

In [ ]:
# Callbacki
cb_early = callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
)
cb_reduce = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1
)

history_lstm = model_lstm.fit(
    X_train_3d, y_train,
    epochs=60,
    batch_size=64,
    validation_data=(X_test_3d, y_test),
    callbacks=[cb_early, cb_reduce],
    verbose=1
)

In [ ]:
# Wykres historii treningu
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_lstm.history['loss'], label='Trening', color='steelblue')
ax1.plot(history_lstm.history['val_loss'], label='Walidacja', color='tomato')
ax1.set_title('Funkcja straty (Binary Cross-Entropy)')
ax1.set_xlabel('Epoka'); ax1.set_ylabel('Strata')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history_lstm.history['accuracy'], label='Trening', color='steelblue')
ax2.plot(history_lstm.history['val_accuracy'], label='Walidacja', color='tomato')
ax2.set_title('Dokładność (Accuracy)')
ax2.set_xlabel('Epoka'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('Przebieg treningu – model LSTM', fontsize=14)
plt.tight_layout()
plt.show()

## Krok 5 – Ocena modelu LSTM

In [ ]:
def evaluate_binary_model(model, X, y_true, dataset_name=''):
    """Ocena modelu binarnego: macierz pomyłek + metryki."""
    y_prob = model.predict(X, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    cm = confusion_matrix(y_true, y_pred)
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)

    print(f'\n=== {dataset_name} ===')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall:    {rec:.4f}')
    print(f'F1-score:  {f1:.4f}')
    print('\nRaport klasyfikacji:')
    print(classification_report(y_true, y_pred, target_names=['Normalny','Arytmia']))

    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['Normalny', 'Arytmia'])
    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'Macierz pomyłek – {dataset_name}', fontsize=12)
    plt.tight_layout()
    plt.show()

    return y_prob

prob_train = evaluate_binary_model(model_lstm, X_train_3d, y_train, 'Dane treningowe')
prob_test  = evaluate_binary_model(model_lstm, X_test_3d,  y_test,  'Dane testowe')

In [ ]:
# Krzywe ROC
fpr_tr, tpr_tr, _ = roc_curve(y_train, prob_train)
fpr_te, tpr_te, _ = roc_curve(y_test,  prob_test)
auc_tr = auc(fpr_tr, tpr_tr)
auc_te = auc(fpr_te, tpr_te)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_tr, tpr_tr, label=f'Trening (AUC = {auc_tr:.4f})', color='steelblue', lw=2)
ax.plot(fpr_te, tpr_te, label=f'Test    (AUC = {auc_te:.4f})', color='tomato',    lw=2)
ax.plot([0,1],[0,1], 'k--', lw=1)
ax.set_xlabel('FPR (False Positive Rate)'); ax.set_ylabel('TPR (True Positive Rate)')
ax.set_title('Krzywa ROC – model LSTM', fontsize=13)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'AUC trening: {auc_tr:.4f}')
print(f'AUC test:    {auc_te:.4f}')

### Dyskusja wyników – Zadanie 3

**Sieć rekurencyjna LSTM vs MLP:**
- Sygnał EKG jest **sekwencją czasową** – każdy punkt zależy od poprzednich. LSTM posiada mechanizm pamięci (bramki: input, forget, output) umożliwiający modelowanie długoterminowych zależności.
- MLP traktuje wszystkie 140 punktów niezależnie, tracąc informację o kolejności próbek.
- LSTM naturalnie zachowuje kontekst czasowy, co przekłada się na lepsze rozpoznawanie wzorców arytmii.

**Obserwacje:**
- Warstwy Dropout (0.3) skutecznie ograniczają przetrenowanie – różnica między stratą treningową a walidacyjną jest niewielka.
- EarlyStopping i ReduceLROnPlateau pozwoliły zatrzymać trening w optymalnym momencie.
- Optymalizator Adam z lr=1e-3 sprawdził się dobrze dzięki adaptacyjnej stopie uczenia.
- F1-score i AUC-ROC są właściwymi metrykami dla niezbalansowanych danych.

---
# Zadanie 4 – Wykrywanie anomalii rytmu serca (Autoenkoder)
---
## Krok 1 – Przygotowanie danych

In [ ]:
# Podział df na train/test 70:30
df_train, df_test = train_test_split(df, test_size=0.3, random_state=42, stratify=df['type'])

# Wydzielenie podzbiorów normalnych i anomalnych
df_train_normal = df_train[df_train['type'] == 0].drop(columns=['type'])
df_train_anom   = df_train[df_train['type'] == 1].drop(columns=['type'])
df_test_normal  = df_test [df_test ['type'] == 0].drop(columns=['type'])
df_test_anom    = df_test [df_test ['type'] == 1].drop(columns=['type'])

print('df_train_normal:', df_train_normal.shape)
print('df_train_anom:  ', df_train_anom.shape)
print('df_test_normal: ', df_test_normal.shape)
print('df_test_anom:   ', df_test_anom.shape)

In [ ]:
# Normalizacja – fitujemy TYLKO na danych treningowych normalnych
scaler_ae = MinMaxScaler()

X_train_normal = scaler_ae.fit_transform(df_train_normal.values)
X_train_anom   = scaler_ae.transform(df_train_anom.values)
X_test_normal  = scaler_ae.transform(df_test_normal.values)
X_test_anom    = scaler_ae.transform(df_test_anom.values)

print('X_train_normal:', X_train_normal.shape)
print('X_train_anom:  ', X_train_anom.shape)
print('X_test_normal: ', X_test_normal.shape)
print('X_test_anom:   ', X_test_anom.shape)

## Krok 2 – Budowa autoenkodera

In [ ]:
input_dim = X_train_normal.shape[1]  # 140

# --- Enkoder ---
inp = keras.Input(shape=(input_dim,), name='input')
x = layers.Dense(64,  activation='relu',    name='enc_1')(inp)
x = layers.Dense(32,  activation='relu',    name='enc_2')(x)
x = layers.Dense(16,  activation='relu',    name='enc_3')(x)
encoded = layers.Dense(8, activation='relu',name='bottleneck')(x)

# --- Dekoder ---
x = layers.Dense(16, activation='relu', name='dec_1')(encoded)
x = layers.Dense(32, activation='relu', name='dec_2')(x)
x = layers.Dense(64, activation='relu', name='dec_3')(x)
decoded = layers.Dense(input_dim, activation='sigmoid', name='output')(x)

autoencoder = keras.Model(inputs=inp, outputs=decoded, name='Autoencoder_EKG')
autoencoder.summary()

## Krok 3 – Kompilacja i trening autoenkodera

In [ ]:
autoencoder.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse'
)

try:
    plot_model(autoencoder, show_shapes=True, show_layer_names=True,
               to_file='ae_model.png', dpi=80)
    display(Image('ae_model.png'))
except Exception:
    print('Wizualizacja modelu niedostępna. Podsumowanie powyżej.')

In [ ]:
cb_es_ae = callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
)
cb_lr_ae = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1
)

# Trening TYLKO na normalnych danych (rekonstrukcja sygnałów normalnych)
history_ae = autoencoder.fit(
    X_train_normal, X_train_normal,
    epochs=100,
    batch_size=32,
    validation_data=(X_test_normal, X_test_normal),
    callbacks=[cb_es_ae, cb_lr_ae],
    verbose=1
)

In [ ]:
# Wykres historii treningu
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history_ae.history['loss'],     label='Trening (normalny)', color='steelblue', lw=2)
ax.plot(history_ae.history['val_loss'], label='Walidacja (normalny)', color='tomato', lw=2)
ax.set_title('Przebieg treningu – Autoenkoder (MSE)', fontsize=13)
ax.set_xlabel('Epoka'); ax.set_ylabel('MSE')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Krok 4 – Ocena autoenkodera

In [ ]:
# Rekonstrukcja i obliczenie błędu MAE
def mae_per_sample(X_orig, X_recon):
    return np.mean(np.abs(X_orig - X_recon), axis=1)

rec_train_normal = autoencoder.predict(X_train_normal, verbose=0)
rec_train_anom   = autoencoder.predict(X_train_anom,   verbose=0)
rec_test_normal  = autoencoder.predict(X_test_normal,  verbose=0)
rec_test_anom    = autoencoder.predict(X_test_anom,    verbose=0)

mae_train_normal = mae_per_sample(X_train_normal, rec_train_normal)
mae_train_anom   = mae_per_sample(X_train_anom,   rec_train_anom)
mae_test_normal  = mae_per_sample(X_test_normal,  rec_test_normal)
mae_test_anom    = mae_per_sample(X_test_anom,    rec_test_anom)

print('Średni MAE – dane treningowe normalne: ', mae_train_normal.mean().round(5))
print('Średni MAE – dane treningowe anomalne: ', mae_train_anom.mean().round(5))
print('Średni MAE – dane testowe normalne:    ', mae_test_normal.mean().round(5))
print('Średni MAE – dane testowe anomalne:    ', mae_test_anom.mean().round(5))

In [ ]:
# Wykresy rekonstrukcji – kilka przykładów normalnych i anomalnych
def plot_reconstruction(X_orig, X_recon, title, color_orig, color_recon, n=4):
    fig, axes = plt.subplots(n, 1, figsize=(12, n*2.5), sharex=True)
    for i, ax in enumerate(axes):
        ax.plot(X_orig[i], color=color_orig, lw=1.5, label='Oryginał')
        ax.plot(X_recon[i], color=color_recon, lw=1.5, linestyle='--', label='Rekonstrukcja')
        ax.set_ylabel(f'Próbka {i+1}')
        ax.grid(True, alpha=0.3)
        if i == 0:
            ax.legend(loc='upper right')
    axes[0].set_title(title, fontsize=13)
    axes[-1].set_xlabel('Czas [próbki]')
    plt.tight_layout()
    plt.show()

plot_reconstruction(X_test_normal, rec_test_normal,
    'Rekonstrukcja EKG – przypadki normalne',
    '#2196F3', '#0D47A1')

plot_reconstruction(X_test_anom, rec_test_anom,
    'Rekonstrukcja EKG – przypadki anomalne (arytmia)',
    '#F44336', '#B71C1C')

In [ ]:
# Histogramy błędu MAE – dane treningowe i testowe
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (mae_n, mae_a, title) in zip(axes, [
    (mae_train_normal, mae_train_anom, 'Dane treningowe'),
    (mae_test_normal,  mae_test_anom,  'Dane testowe')
]):
    ax.hist(mae_n, bins=50, alpha=0.6, color='#2196F3', label='Normalne', density=True)
    ax.hist(mae_a, bins=50, alpha=0.6, color='#F44336', label='Anomalne', density=True)
    ax.set_xlabel('MAE rekonstrukcji')
    ax.set_ylabel('Gęstość')
    ax.set_title(f'Histogram MAE – {title}', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Rozkład błędu MAE rekonstrukcji autoenkodera', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Histogram zawężony do zakresu przejściowego (dane treningowe)
# Znajdź zakres gdzie obie klasy się nakładają
mae_min = max(mae_train_normal.min(), mae_train_anom.min())
mae_max = min(mae_train_normal.max(), mae_train_anom.max())
# Trochę poszerz zakres
margin = (mae_max - mae_min) * 0.3
x_min = max(0, mae_min - margin)
x_max = mae_max + margin

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(mae_train_normal[(mae_train_normal >= x_min) & (mae_train_normal <= x_max)],
        bins=40, alpha=0.6, color='#2196F3', label='Normalne', density=True)
ax.hist(mae_train_anom[(mae_train_anom >= x_min) & (mae_train_anom <= x_max)],
        bins=40, alpha=0.6, color='#F44336', label='Anomalne', density=True)
ax.set_xlim(x_min, x_max)
ax.set_xlabel('MAE rekonstrukcji')
ax.set_ylabel('Gęstość')
ax.set_title('Histogram MAE – strefa przejściowa (dane treningowe)', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Zakres progu odcięcia: [{x_min:.5f}, {x_max:.5f}]')

In [ ]:
# Wykres accuracy vs próg odcięcia (dane treningowe)
thresholds = np.linspace(x_min, x_max, 200)

y_true_train = np.concatenate([
    np.zeros(len(mae_train_normal)),
    np.ones(len(mae_train_anom))
])
mae_all_train = np.concatenate([mae_train_normal, mae_train_anom])

accuracies = []
for t in thresholds:
    y_pred = (mae_all_train >= t).astype(int)
    accuracies.append(accuracy_score(y_true_train, y_pred))

best_idx   = np.argmax(accuracies)
best_thresh = thresholds[best_idx]
best_acc   = accuracies[best_idx]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, accuracies, color='steelblue', lw=2)
ax.axvline(best_thresh, color='red', linestyle='--', label=f'Optymalny próg = {best_thresh:.5f}')
ax.scatter([best_thresh], [best_acc], color='red', s=80, zorder=5)
ax.set_xlabel('Próg odcięcia MAE')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy vs próg odcięcia – dane treningowe', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Optymalny próg odcięcia: {best_thresh:.5f}')
print(f'Maksymalna accuracy:     {best_acc:.4f}')

In [ ]:
# Macierze pomyłek i metryki dla optymalnego progu
threshold = best_thresh

y_true_test = np.concatenate([
    np.zeros(len(mae_test_normal)),
    np.ones(len(mae_test_anom))
])
mae_all_test = np.concatenate([mae_test_normal, mae_test_anom])

def evaluate_autoencoder(mae_all, y_true, threshold, dataset_name):
    y_pred = (mae_all >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)

    print(f'\n=== {dataset_name} (próg = {threshold:.5f}) ===')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall:    {rec:.4f}')
    print(f'F1-score:  {f1:.4f}')
    print('\nRaport klasyfikacji:')
    print(classification_report(y_true, y_pred, target_names=['Normalny', 'Arytmia']))

    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['Normalny', 'Arytmia'])
    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'Macierz pomyłek – {dataset_name}', fontsize=12)
    plt.tight_layout()
    plt.show()
    return y_pred

_ = evaluate_autoencoder(mae_all_train, y_true_train, threshold, 'Dane treningowe')
_ = evaluate_autoencoder(mae_all_test,  y_true_test,  threshold, 'Dane testowe')

In [ ]:
# Krzywe ROC dla autoenkodera
fpr_tr_ae, tpr_tr_ae, _ = roc_curve(y_true_train, mae_all_train)
fpr_te_ae, tpr_te_ae, _ = roc_curve(y_true_test,  mae_all_test)
auc_tr_ae = auc(fpr_tr_ae, tpr_tr_ae)
auc_te_ae = auc(fpr_te_ae, tpr_te_ae)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_tr_ae, tpr_tr_ae, label=f'Trening (AUC = {auc_tr_ae:.4f})', color='steelblue', lw=2)
ax.plot(fpr_te_ae, tpr_te_ae, label=f'Test    (AUC = {auc_te_ae:.4f})', color='tomato', lw=2)
ax.plot([0,1],[0,1], 'k--', lw=1)
ax.set_xlabel('FPR (False Positive Rate)')
ax.set_ylabel('TPR (True Positive Rate)')
ax.set_title('Krzywa ROC – Autoenkoder', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'AUC trening: {auc_tr_ae:.4f}')
print(f'AUC test:    {auc_te_ae:.4f}')

## Krok 5 – Dyskusja wyników – Zadanie 4

### Czy model jest przydatny do detekcji arytmii?

Autoenkoder trenowany wyłącznie na normalnych rytmach serca nauczył się ich kompaktowej reprezentacji. Kiedy rekonstruuje sygnał arytmiczny, popełnia wyższy błąd MAE – bo nie "nauczył się" wzorców anomalnych. Ta różnica w błędzie rekonstrukcji stanowi podstawę do detekcji.

- **AUC > 0.9** (typowo) wskazuje na dobrą separację klas bez użycia jakichkolwiek etykiet podczas treningu
- Metoda jest szczególnie wartościowa, gdy nie dysponujemy etykietami dla danych anomalnych

### Tendencja do przetrenowania

- Autoenkoder z 4+4 warstwami gęstymi (64→32→16→8→16→32→64→140) jest stosunkowo mały – ryzyko przetrenowania jest niskie
- EarlyStopping skutecznie zatrzymuje trening, gdy walidacja przestaje się poprawiać
- Dodanie warstw Dropout (np. 0.1–0.2) może dodatkowo zmniejszyć przetrenowanie, ale też nieco pogorszyć jakość rekonstrukcji normalnych sygnałów
- Wąskie gardło (bottleneck = 8 neuronów) wymusza kompresję i jest naturalnym regularyzatorem

### Algorytm optymalizacyjny

- **Adam (lr=1e-3)** – najlepiej sprawdza się w praktyce dla autoenkoderów: szybka zbieżność i adaptacyjna stopa uczenia
- RMSprop daje podobne wyniki, ale wolniej
- SGD z momentum wymaga starannego doboru lr i większej liczby epok

### Wybór progu odcięcia

Próg wyznaczono na podstawie maksymalizacji accuracy na danych treningowych. W zastosowaniach medycznych warto rozważyć **recall (czułość)** jako priorytetową metrykę – lepiej błędnie zaklasyfikować zdrowego jako chorego, niż przeoczyć arytmię.